# 彩色装箱问题 (CBPP)

**类别:** 装箱

来源: [https://www.hexaly.com/templates/colored-bin-packing-problem-cbpp](https://www.hexaly.com/templates/colored-bin-packing-problem-cbpp)


## 问题描述

**在彩色装箱问题 (BPP)** 中,一组具有已知重量和颜色的物品必须分配到具有相同容量的箱子中。每个物品必须被放入恰好一个箱子内。每个箱子内物品的总重量不能超过其容量,且箱子内的所有物品必须具有不同的颜色。目标是最小化所用箱子的数量。该问题是 [装箱问题 (BPP)](https://www.hexaly.com/docs/last/exampletour/binpacking.html) 的一个变种,因此是 NP 难的。

	

### 学到的要点

- 添加 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模箱子的内容
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算箱子的总重量以及箱子中给定颜色的元素数量


## 数据

所提供的彩色装箱问题 (CBPP) 实例来自 AI-Q2 数据集,它是 [BPPLIB](http://or.dei.unibo.it/library/bpplib) 中 Augmented IRUP (AI) 实例的一个改编版本,其中每个物品的颜色被随机选择。数据文件的格式如下:

- 第一行:物品数量、颜色数量、箱子容量
- 对每个物品,给出其重量和颜色


## 模型

彩色装箱问题 (CBPP) 的 Hexaly 模型使用 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对于每个箱子,我们定义一个集合变量来表示分配到该箱子的物品集合。我们将集合变量约束为形成一个 [partition](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition),以确保每个物品恰好属于一个箱子。

我们使用集合上的可变参 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)(返回与任何物品索引相关联的重量)来计算箱子的总重量。注意,该求和中项的数量在搜索过程中是变化的,集合的大小也随之变化。然后我们可以将总重量约束为小于箱子容量。

我们还使用集合上的可变参 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)(若物品具有给定颜色则返回 1,否则返回 0)来计算箱子中给定颜色的元素数量。该求和中项的数量在搜索过程中也是变化的,集合的大小也随之变化。然后我们可以将每个求和约束为小于 1。

如果一个箱子至少包含一个物品,则该箱子被实际使用。利用 **count** 算子(返回集合中元素的数量),我们可以检查每个箱子是否被实际使用,然后计算出所用箱子的总数。

模型计算最优箱子数量的简单上下界。它仅定义 nbMaxBins 个集合变量,并使用 [hxObjectiveThreshold](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#hxObjectiveThreshold) 在达到 nbMinBins 个箱子的解时停止搜索。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math

if len(sys.argv) < 2:
    print("Usage: python bin_packing.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

with hexaly.optimizer.HexalyOptimizer() as optimizer:
    # Read instance data
    file_it = iter(read_integers(sys.argv[1]))
    nb_items = next(file_it)
    nb_colors = next(file_it)
    bin_capacity = next(file_it)
    weights_data = []
    colors_data = []

    for _ in range(nb_items):
        weights_data.append(next(file_it))
        colors_data.append(next(file_it))

    nb_min_bins = int(math.ceil(sum(weights_data) / float(bin_capacity)))
    nb_max_bins = nb_items

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Set decision : bins[k] represents the items in bin k
    bins = [model.set(nb_items) for _ in range(nb_max_bins)]

    # Transform bins list into HxExpression
    bins_array = model.array(bins)

    # Each item must be in exactly one bin
    model.constraint(model.partition(bins))

    # Create an array and a function to retrieve item's weight and color
    weights = model.array(weights_data)
    weight_lambda = model.lambda_function(lambda i: weights[i])
    colors = model.array(colors_data)
    color_lambda = [model.lambda_function(lambda j: colors[j] == i ) for i in range(nb_colors)]

    # Weight constraint for each bin
    bin_weights = [model.sum(b, weight_lambda) for b in bins]
    for w in bin_weights:
        model.constraint(w <= bin_capacity)
    
    # At most one item of each color per bin
    nb_colors_per_bin = [[model.sum(b, color_lambda_i) for color_lambda_i in color_lambda] for b in bins]
    for bin in nb_colors_per_bin:
        for nb_color in bin:
            model.constraint(nb_color <= 1)


    # Bin k is used if at least one item is in it
    bins_used = [model.count(b) > 0 for b in bins]

    # Count the used bins
    total_bins_used = model.sum(bins_used)

    # Minimize the number of used bins
    model.minimize(total_bins_used)
    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    # Stop the search if the lower threshold is reached
    optimizer.param.set_objective_threshold(0, nb_min_bins)

    optimizer.solve()

 # Write the solution in a file
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            for k in range(nb_max_bins):
                if bins_used[k].value > 0:
                    f.write("Bin weight: %d | Items: " % bin_weights[k].value)
                    for e in bins[k].value:
                        f.write("%d " % e)
                    f.write("\n")
